# Compare models × test sets × approaches (workstream C)

**Three approaches, same candidates:**

| approach | what decides | score |
|---|---|---|
| **algorithmic** | `ThresholdDetector` proposal + its peak temperature. No ML. | peak °C |
| **CNN alone** | the CNN verifier at the candidate. No temperature gate, no hot-blob bypass. | p(fire) |
| **full pipeline** | what `FusedDetector` emits: CNN p(fire), raised to 0.9 when the hot-blob bypass fires (decision 0003) | p(fire) |

**Six test views** (none is ever trained on; details and the reasoning in §1):

| view | what | labels |
|---|---|---|
| `nadir` | FLAME 3 NADIR **plot 1** (Hanna Hammock, straight down, 15.7 cm/px) | from each pixel's history; point-level hard view |
| `handlabels` | your RGB-judged Sycan blobs, 80–200 °C | you (`unsure` dropped) |
| `composite` | synthetic fires on real Sycan No-Fire frames, 4 held-out cells | exact |
| `cvsubset` | Sycan CV subset test groups (oblique) | weak: clipped/≥200 °C = fire, No-Fire frames = not fire, rest ignored |
| `all_with_hand` / `all_no_hand` | the above pooled, with / without your hand labels | |

Plus `nadir_proposals`: NADIR at proposal level (~99 % fire, so recall only).

**Models:** the shipped composite model vs `mix_v2` (trained on NADIR plot 2 + composites + Sycan weak labels, with hard examples).

Same code as the CLI: `training/compare.py`, `training/eval_sets.py`.

## 0. Configuration

In [ ]:
CONFIG = dict(
    models={
        "composite (shipped)": "perception/models/fire_cnn_flame3comp.onnx",
        "mix_v2 (NADIR + Sycan + composite)": "perception/models/fire_cnn_mix.onnx",
    },
    views=("nadir", "handlabels", "composite", "cvsubset", "all_with_hand", "all_no_hand", "nadir_proposals"),
    labels="exclude/labels/labels_DoqCA.csv",     # hand labels (one labeller); 'unsure' dropped
    positive=("active", "smoulder"),               # what counts as fire in the hand labels
    saturation_c=550.0,                            # default gain-mode saturation; per-frame saturation_c overrides it
    # drill-down (section 4)
    focus_model="mix_v2 (NADIR + Sycan + composite)",
    focus_view="all_with_hand",
    bands=dict(algorithmic=(60.0, 150.0), model=None, pipeline=None),   # None = model sidecar (reject, confirm)
    run_object_level=False,                        # section 5: frame/object metrics on the manifest views (slower)
)

In [ ]:
import os, sys
from pathlib import Path
if not Path("perception").exists() and Path("../perception").exists():
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

from perception.datasets import MASK_IGNORE, load_frame, load_manifest, load_mask
from perception.detectors import HotBypassRule, load_classifier
from perception.triage import TriageBands, suggest_bands
from training.compare import APPROACHES, compare, default_bands, default_search_thresholds, markdown
from training.compare_plots import (CLASS_COLORS, matrix_markdown, plot_curves, plot_errors, plot_matrix,
                                    plot_reliability, plot_sweep, triage_panel, triage_summary)
from training.eval_sets import Paths, score_views

C = CONFIG
PATHS = Paths(labels=C["labels"])
BYPASS = HotBypassRule(saturation_c=C["saturation_c"])
CLF = {name: load_classifier(p) for name, p in C["models"].items() if Path(p).exists()}
for name, p in C["models"].items():
    if name not in CLF:
        print(f"missing model {p}: skipped")
for name, clf in CLF.items():
    print(f"{name}: reject {clf.reject_threshold:.3f} · search {clf.threshold:.3f} · confirm {clf.confirm_threshold:.3f}")

## 1. How each dataset was built (and why)

### NADIR: labels from each pixel's own history (no hand labelling needed)
The georeferenced TIFFs of a plot are stabilized onto one map grid, so every pixel has a temperature history: ~30 °C unburned → 300–500 °C as the front passes → cools to a **char plateau ~43–45 °C** (black char in the sun is warmer than unburned ground). One frame can't tell a 70 °C cooling ember from sun-warmed char; the history can. Rules (`training/build_nadir.py`, all `[ASSUMPTION]` defaults):

* **fire**: ≥ 120 °C now (the sun can't do that), **or** burned already and still ≥ 25 °C above where the pixel ends up (its post-peak floor) = residual heat
* **not fire**: never above 60 °C so far and < 50 °C now (unburned), **or** burned, ≥ 20 frames after its peak, within 8 °C of its floor, floor near the char plateau (burnt out)
* **ignore**: everything else (pre-heating ahead of the front, mid-cooling, frame borders). Excluded from training and scoring.

Reshaping: plot 1 = **test**, plot 2 (two separately registered parts) = **train**; every 3rd frame (they're 3–5 s apart); CV groups = 30-frame time blocks; nodata borders filled with the frame median and marked ignore; GSD from the GeoTIFF transform; high-gain frames (clip ~197 °C) detected per frame.

In [ ]:
nad = load_manifest(PATHS.nadir)
p1 = [r for r in nad if r.source == "flame3_nadir:p1"]
summ = np.load("exclude/data/flame3_nadir_prepared/summary_p1.npz")
from matplotlib.colors import ListedColormap
mask_cmap = ListedColormap([CLASS_COLORS["not_fire"], CLASS_COLORS["fire"], "#d9d8d3"])
fig, ax = plt.subplots(2, 4, figsize=(17, 7.5))
for k, i in enumerate((20, 60, 75, 98)):
    r = p1[i]
    ax[0, k].imshow(load_frame(r), cmap="inferno", vmin=25, vmax=250); ax[0, k].set_title(Path(r.frame_path).stem, fontsize=9)
    ax[1, k].imshow(np.load(r.mask_path), cmap=mask_cmap, vmin=0, vmax=2, interpolation="nearest")
    ax[1, k].set_title("blue not fire · orange fire · grey ignore", fontsize=8)
for a in ax.flat: a.axis("off")
plt.tight_layout(); plt.show()

# pixel histories (from the written frames, every 3rd) with the rule thresholds
stack = np.stack([load_frame(r) for r in p1])
peak, floor = summ["peak"], summ["floor"]
rng = np.random.default_rng(1)
fig, ax = plt.subplots(figsize=(11, 3.8))
ys, xs = np.nonzero(peak >= 250)
for i in rng.choice(len(ys), 5, replace=False):
    ax.plot(np.arange(len(p1)) * 3, stack[:, ys[i], xs[i]], lw=1.5)
ys, xs = np.nonzero(peak < 50)
for i in rng.choice(len(ys), 2, replace=False):
    ax.plot(np.arange(len(p1)) * 3, stack[:, ys[i], xs[i]], color="#6b6a64", lw=1)
ax.axhline(120, ls="--", color="#6b6a64", lw=1); ax.text(2, 125, "fire ≥ 120 °C", fontsize=8)
ax.axhline(float(summ["plateau"]), ls=":", color="#6b6a64", lw=1); ax.text(2, float(summ["plateau"]) + 4, "char plateau", fontsize=8)
ax.set_xlabel("frame"); ax.set_ylabel("°C"); ax.set_ylim(15, 500); ax.set_title("NADIR plot 1: pixel histories (colour = burned pixels, grey = unburned)", loc="left", fontsize=10)
plt.show()

### NADIR test view is point-level
At proposal level NADIR is ~99 % fire: the proposer keeps the 64 hottest blobs per frame, and one big blob touching any ember counts as fire, so the useful negatives (warm char) never get proposed. So `nadir` samples, per test frame, **20 warm not-fire points** (≥ 8 °C above the frame median, ≥ 1.2 m from any fire *or ignore* pixel) and **20 fire points** (half from the cooler half = residual heat), scored by all three approaches. `nadir_proposals` keeps the proposal view as a recall check.

### Sycan CV subset: weak labels that only say what the dataset's rule makes certain
FLAME 3 labels whole frames by max temperature. At blob level that supports two statements, so the mask says only those: **No-Fire frame → not fire** (incl. sun-warmed rocks/logs the proposer finds); **Fire frame, pixel ≥ 200 °C or at the gain clip (187.4 °C high gain) → fire**; everything else in a Fire frame → **ignore** (the 80–200 °C band is what your hand labels are for). Split by group; No-Fire groups keep the composite split (they're composite backgrounds); the 117 train-group frames holding a hand-labelled candidate are **never trained on** (`split = holdout`).

### Composites: unchanged
Synthetic fires and decoys inserted into real Sycan No-Fire frames (`training/composite.py`). The only view with exact masks and balanced hard negatives.

### Training mix (`mix_v2`)
NADIR plot 2 + composite train + Sycan weak train (703 frames, 40 groups, 24,220 patches, 56 % fire). Two changes to patch sampling were needed (`training/patch_data.py`): **mask-centred positives capped at 6 components per frame** (real masks have thousands of components: the first run was 92 % positive), and **hard examples straight from the masks**: 12 warm not-fire points and 4 cooler-half fire points per frame.

In [ ]:
cv = [r for r in load_manifest(PATHS.cvsubset, split="test") if r.label][:1] + \
     [r for r in load_manifest(PATHS.composite, split="test") if r.mask_path][:1]
fig, ax = plt.subplots(1, 4, figsize=(17, 3.8))
for k, r in enumerate(cv):
    t = load_frame(r); raw = np.load(r.mask_path) if r.mask_path.endswith(".npy") else load_mask(r).astype(np.uint8)
    ax[2 * k].imshow(t, cmap="inferno", vmin=np.percentile(t, 1), vmax=max(np.percentile(t, 99.9), 60)); ax[2 * k].set_title(("Sycan Fire frame" if k == 0 else "composite") + f" · {Path(r.frame_path).stem}", fontsize=9)
    ax[2 * k + 1].imshow(raw, cmap=mask_cmap, vmin=0, vmax=2, interpolation="nearest"); ax[2 * k + 1].set_title("weak mask" if k == 0 else "exact mask", fontsize=9)
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

## 2. Score every model on every view
Takes ~1–2 min per model; results are cached in `CACHE` so re-running later cells is instant.

In [ ]:
try:
    CACHE
except NameError:
    CACHE = {}
RESULTS, SCORED = {}, {}
for name, clf in CLF.items():
    SCORED[name] = score_views(clf, C["views"], PATHS, BYPASS, tuple(C["positive"]), cache=CACHE.setdefault(name, {}))
    RESULTS[name] = {v: compare(sc, default_bands(clf), default_search_thresholds(clf), n_boot=300)
                     for v, sc in SCORED[name].items()}
    print(name, {v: len(sc) for v, sc in SCORED[name].items()})

## 3. The matrix
**precision / recall / F1** = the single search threshold each approach uses on the drone today (algorithmic 60 °C; CNN = sidecar `threshold`). **missed** = fires in NOT_FIRE and **review** = share in REVIEW at the sidecar bands. AUC 0.5 = coin flip.

Read each view for what it can show: `cvsubset` and `nadir` separate on temperature by construction (labels only cover clear cases), so they mainly test whether the CNN **drops real fires** (recall) or **calls warm char/rocks fire** (precision). `composite` is the balanced hard test; `handlabels` is the 80–200 °C band (negatives questionable, see #19).

In [ ]:
display(Markdown(matrix_markdown(RESULTS)))
plot_matrix(RESULTS, "f1"); plt.tight_layout(); plt.show()
plot_matrix(RESULTS, "recall"); plt.tight_layout(); plt.show()
plot_matrix(RESULTS, "roc_auc"); plt.tight_layout(); plt.show()

## 4. Drill down: one model, one view
Set `focus_model` / `focus_view` in the config, then move the sliders. Grey = REVIEW band.

In [ ]:
FM, FV = C["focus_model"], C["focus_view"]
sc, clf = SCORED[FM][FV], CLF[FM]
BANDS = default_bands(clf)
for a, b in C["bands"].items():
    if b is not None:
        BANDS[a] = TriageBands(*b)
display(Markdown(markdown(compare(sc, BANDS, default_search_thresholds(clf), n_boot=300), f"{FV} · {FM}")))

In [ ]:
triage_panel(sc, "algorithmic", BANDS["algorithmic"])

In [ ]:
triage_panel(sc, "model", BANDS["model"])

In [ ]:
triage_panel(sc, "pipeline", BANDS["pipeline"])

In [ ]:
plot_curves(sc); plt.tight_layout(); plt.show()
plot_sweep(sc, "model", max_missed=0.02); plt.tight_layout(); plt.show()

Suggested bands (fires missed ≤ 1 %, FIRE precision ≥ 0.98). **On a test view this fits the test set**: use it to see what the data asks for, then set bands from CV.

In [ ]:
for a in APPROACHES:
    try:
        sb = suggest_bands(sc.y, sc.scores(a), max_missed_fire_rate=0.01, min_fire_precision=0.98)
        print(triage_summary(sc, a, sb["bands"]) + ("" if sb["high"]["met_target"] else "  (precision target not reachable)"))
    except ValueError as e:
        print(a, e)
plot_reliability(sc, "model"); plt.tight_layout(); plt.show()

In [ ]:
r = compare(sc, BANDS, default_search_thresholds(clf), n_boot=100)
rows = ["| peak band | approach | fire | not fire | fires missed | review rate | FIRE precision |", "|---|---|---|---|---|---|---|"]
f = lambda x: "-" if x != x else f"{x:.3f}"
for band, per in r["by_temp_band"].items():
    for a, t in per.items():
        rows.append(f"| {band} | {a} | {t['n_fire']} | {t['n_not_fire']} | {t['counts']['NOT_FIRE']['fire']} ({f(t['missed_fire_rate'])}) | {f(t['review_rate'])} | {f(t['fire_precision'])} |")
display(Markdown("\n".join(rows)))

Mistakes (thermal crop; RGB below for hand labels).

In [ ]:
for a in ("model", "pipeline"):
    for kind in ("missed", "false_fire"):
        if plot_errors(sc, a, BANDS[a], kind=kind, n=8) is not None:
            plt.show()

## 5. Frame and object level (manifest views)
The deployed decision at frame/object level (`perception/evaluate.py`, ignore regions excluded). Set `run_object_level=True`.

In [ ]:
if C["run_object_level"]:
    from perception.evaluate import evaluate_pipeline, _markdown
    for view in ("nadir", "composite", "cvsubset"):
        recs = load_manifest(getattr(PATHS, view), split="test")
        res = evaluate_pipeline(recs, {"cnn": CLF[FM]}, BYPASS)
        display(Markdown(_markdown(res, f"{view} · {FM}", {"cnn": CLF[FM]})))